In [ ]:
try:
    import firedrake
except ImportError:
    !wget "https://fem-on-colab.github.io/releases/firedrake-install-release-real.sh" -O "/tmp/firedrake-install.sh" && bash "/tmp/firedrake-install.sh"
    import firedrake

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
from firedrake import *
import matplotlib.pyplot as plt

import numpy as np

In [ ]:
# numpy array from Firedrake Matrix object (output of assemble of bilinear form)
def mat2numpy(A):
  return A.petscmat.getValues(range(0, A.petscmat.getSize()[0]), range(0,  A.petscmat.getSize()[1]))
# numpy array from Firedrake objects containing vectors of d.o.f.'s
# (Functions or output of assemble of linear form)
def dofvec2numpy(b):
  return b.dat.data

# Linear solvers in Firedrake / PETSc

We consider the diffusion-reaction problem

\begin{equation*}
\begin{cases}
- \Delta u + u = f & {\rm in} \ \Omega=(0,1)^2, \\
u = 0 & {\rm on} \ \partial\Omega,
\end{cases}
\end{equation*}
with $f(x,y)=(1+2\pi^2)\sin(\pi x)\sin(\pi y)$ and exact solution $u_\text{ex}(x,y)=\sin(\pi x)\sin(\pi y)$.

In [ ]:
N = 10
# Build the mesh
mesh = UnitSquareMesh(N, N, diagonal='left')

# Plot it
fig, ax = plt.subplots()
triplot(mesh, axes=ax)
ax.legend()
ax.axis('equal')

# Define a discrete function space
V = FunctionSpace(mesh, 'P', 1)

# Define trial and test functions as belonging to the space
u = TrialFunction(V)
v = TestFunction(V)

# Data
x = SpatialCoordinate(mesh)
u_ex = Function(FunctionSpace(mesh, 'P', 2))    # higher-order FE space to avoid spoiling from interpolation error
u_ex.interpolate(sin(pi*x[0])*sin(pi*x[1]))
f = (1+2*pi*pi)*sin(pi*x[0])*sin(pi*x[1])
bcs = [ DirichletBC(V, Constant(0.0), (1,2,3,4)) ]

# Define the variational problem: bilinear form and rhs
a = dot(grad(u), grad(v)) * dx + u*v*dx
L = f*v*dx

In [ ]:
uh = Function(V)

# Define structures to assemble matrix and vector (does not actually perform the assembly: this is done at the first call of a solver)
vpb = LinearVariationalProblem(a, L, uh, bcs=bcs)

In [ ]:
from time import perf_counter

default_solver =  LinearVariationalSolver(vpb)
# NOT NECESSARILY THE ONE REPORTED HERE:
# https://www.firedrakeproject.org/solving-interface.html#default-solver-options

parameters = {'ksp_type': 'preonly', "pc_type": "lu",
                             "pc_factor_mat_solver_type": "mumps"}
direct_solver =  LinearVariationalSolver(vpb, solver_parameters=parameters)

parameters = {'ksp_type': 'gmres', 'pc_type': 'none',
              'ksp_rtol': 1.e-8, 'ksp_max_it': 10000}
gmres_solver =  LinearVariationalSolver(vpb, solver_parameters=parameters)

parameters = {'ksp_type': 'cg', 'pc_type': 'none',
              'ksp_rtol': 1.e-8, 'ksp_max_it': 10000}
cg_solver =  LinearVariationalSolver(vpb, solver_parameters=parameters)

uh.assign(0)    # initialization to avoid cross-effects from other solvers
t0 = perf_counter()
default_solver.solve()
print('DEFAULT SOLVER    -   elapsed time =', perf_counter() - t0, 's    -    # iter =', default_solver.snes.ksp.getIterationNumber())
print('                  -   error =', errornorm(u_ex, uh, 'H1'))

uh.assign(0)    # initialization to avoid cross-effects from other solvers
t0 = perf_counter()
direct_solver.solve()
print('DIRECT SOLVER     -   elapsed time =', perf_counter() - t0, 's    -    # iter =', direct_solver.snes.ksp.getIterationNumber())
print('                  -   error =', errornorm(u_ex, uh, 'H1'))

uh.assign(0)    # initialization to avoid cross-effects from other solvers
t0 = perf_counter()
gmres_solver.solve()
print('GMRES SOLVER      -   elapsed time =', perf_counter() - t0, 's    -    # iter =', gmres_solver.snes.ksp.getIterationNumber())
print('                  -   error =', errornorm(u_ex, uh, 'H1'))

uh.assign(0)    # initialization to avoid cross-effects from other solvers
t0 = perf_counter()
cg_solver.solve()
print('CG SOLVER         -   elapsed time =', perf_counter() - t0, 's    -    # iter =', cg_solver.snes.ksp.getIterationNumber())
print('                  -   error =', errornorm(u_ex, uh, 'H1'))

In [ ]:
# Plot the numerical solution
fig, ax = plt.subplots()
q = tripcolor(uh, axes=ax)
fig.colorbar(q)

In [ ]:
# check default parameters
from firedrake.petsc import DEFAULT_KSP_PARAMETERS
print('default parameters:',DEFAULT_KSP_PARAMETERS,'\n')

# check stop criterion
A = mat2numpy(assemble(a, bcs=bcs))
b = dofvec2numpy(assemble(L, bcs=bcs))
u_array = dofvec2numpy(uh)
res = b - (A @ u_array)     # numpy matrix-product operator: @
print('CG SOLUTION  -   relative residual =', np.linalg.vector_norm(res)/np.linalg.vector_norm(b))

# further details on iterative solver
parameters.update({"ksp_monitor": None, 'ksp_monitor_true_residual': None})
cg_solver =  LinearVariationalSolver(vpb, solver_parameters=parameters)
uh.assign(0)    # initialization to avoid cross-effects from other solvers
t0 = perf_counter()
cg_solver.solve()

Preconditioning

In [ ]:
parameters = {'ksp_type': 'gmres', 'pc_type': 'none',
              'ksp_rtol': 1.e-8, 'ksp_max_it': 10000}
gmres_solver =  LinearVariationalSolver(vpb, solver_parameters=parameters)

parameters.update({'pc_type': 'ilu'})
pgmres_solver =  LinearVariationalSolver(vpb, solver_parameters=parameters)

uh.assign(0)    # initialization to avoid cross-effects from other solvers
t0 = perf_counter()
gmres_solver.solve()
print('GMRES SOLVER      -   elapsed time =', perf_counter() - t0, 's    -    # iter =', gmres_solver.snes.ksp.getIterationNumber())
print('                  -   error =', errornorm(u_ex, uh, 'H1'))

uh.assign(0)    # initialization to avoid cross-effects from other solvers
t0 = perf_counter()
pgmres_solver.solve()
print('P-GMRES SOLVER    -   elapsed time =', perf_counter() - t0, 's    -    # iter =', pgmres_solver.snes.ksp.getIterationNumber())
print('                  -   error =', errornorm(u_ex, uh, 'H1'))

# Ex.1 - Monolithic Stokes system

### Cavity problem

\begin{equation*}
\begin{cases}
- \Delta \boldsymbol{u} + \nabla  p  = \boldsymbol{0} & {\rm in} \ \Omega=(0,1)^2, \\
{\rm div}\,\boldsymbol{u} = 0 & {\rm in} \ \Omega, \\
\boldsymbol{u} = \boldsymbol{g}_\text{D} & {\rm on} \ \Gamma_4.\\
\boldsymbol{u} = \boldsymbol{0} & {\rm on} \ \partial\Omega\setminus\Gamma_4, \\
\end{cases}
\end{equation*}

with $\boldsymbol{g}_\text{D} = 1\boldsymbol{i}$.

### Points 1-2 : first solution

In [ ]:
# Build the mesh
n = 20
mesh = UnitSquareMesh(n, n)

fig, ax = plt.subplots()
triplot(mesh, axes=ax)
ax.legend()

In [ ]:
# Function spaces
V = VectorFunctionSpace(mesh, 'P', 2)
Q = FunctionSpace(mesh, 'P', 1)
W = MixedFunctionSpace([V, Q])
print('Ndofs - velocity :',V.dim(),', pressure :',Q.dim(),', total :',W.dim())

# Finite element functions
u, p = TrialFunctions(W)
v, q = TestFunctions(W)

In [ ]:
# Boundary conditions (strong)
bc4 = DirichletBC(W.sub(0), Constant((1., 0.)), 4)
bcRest = DirichletBC(W.sub(0), Constant((0., 0.)), [1,2,3])
bcs = (bc4, bcRest)

# Variational formulation
a = inner(grad(u), grad(v)) * dx - div(v) * p * dx + q * div(u) * dx
L = inner(Constant((0.0,0.0)), v) * dx
  # Dummy rhs (=0) to ensure that the solve recognize a==L as a linear problem

# Null space for fully Dirichlet conditions
nsp = MixedVectorSpaceBasis(
    W, [W.sub(0), VectorSpaceBasis(constant=True)]
)

# Solution (NB: do not use the same name u,v,p,q of the trial/test functions)
from time import perf_counter
wh = Function(W)
vpb = LinearVariationalProblem(a, L, wh, bcs)

In [ ]:
parameters = {'ksp_type': 'gmres', 'pc_type': 'ilu',
              'ksp_rtol': 1.e-8, 'ksp_max_it': 10000}
solver =  LinearVariationalSolver(vpb, nullspace=nsp, solver_parameters=parameters)
t0 = perf_counter()
solver.solve()
print('elapsed time = ', perf_counter() - t0, 's    -    # iter = ', solver.snes.ksp.getIterationNumber())
uh, ph = wh.subfunctions

In [ ]:
fig, ax = plt.subplots()
col = tripcolor(ph, axes=ax)
plt.colorbar(col)
plt.title('pressure')
fig, ax = plt.subplots()
#triplot(mesh, axes=ax)
col = quiver(uh, axes=ax)
plt.colorbar(col)
plt.title('velocity')

### Point 3. Block diagonal preconditioner

Notation for `fieldsplit`

(example with simpler eq. in https://www.firedrakeproject.org/demos/saddle_point_systems.py.html)

$$
\Sigma = \begin{bmatrix} \Sigma_{00} & \Sigma_{01} \\ \Sigma_{10} & \Sigma_{11} \end{bmatrix} = \begin{bmatrix} A & B^T \\ -B & 0 \end{bmatrix}, \qquad P = \begin{bmatrix} P_0 & 0 \\ 0 & P_1 \end{bmatrix} = \begin{bmatrix} A & 0 \\ 0 & \frac{1}{\nu}M_p \end{bmatrix}
$$

$$
\Sigma\begin{bmatrix} \mathbf{U} \\\boldsymbol{\Pi} \end{bmatrix}=\begin{bmatrix} \mathbf{F} \\\mathbf{0} \end{bmatrix}
\longrightarrow
P^{-1}\Sigma\begin{bmatrix} \mathbf{U} \\\boldsymbol{\Pi} \end{bmatrix}=P^{-1}\begin{bmatrix} \mathbf{F} \\\mathbf{0} \end{bmatrix}
\leftrightarrow
\begin{bmatrix} P_0^{-1}A & P_0^{-1}B^T \\ -P_1^{-1}B & 0 \end{bmatrix}\begin{bmatrix} \mathbf{U} \\\boldsymbol{\Pi} \end{bmatrix}=\begin{bmatrix} P_0^{-1}\mathbf{F} \\\mathbf{0} \end{bmatrix}
$$
so for each of the components $P_0, P_1$ of the preconditioner, we need to be able to "apply its inverse" (or an approximation of it).

**N.B.** We **never** actually compute the inverse of a matrix: to "apply the inverse" of a matrix means to solve the associated linear system.

In [ ]:
# Create preconditioner form
pc_form = inner(grad(u), grad(v)) * dx + p * q * dx

parameters = {'ksp_type': 'gmres',
              'ksp_rtol': 1.e-8, 'ksp_max_it': 10000,
              "pc_type": "fieldsplit",                                              # use block structure of monolithic matrix
              "pc_fieldsplit_type": "additive",                                     # use a block diagonal preconditioner (taken from pc_form)
              "fieldsplit_0_ksp_type": "preonly", "fieldsplit_0_pc_type": "lu",     # block P0 is "inverted" by LU factorization
              "fieldsplit_1_ksp_type": "preonly", "fieldsplit_1_pc_type": "lu"}     # block P1 is "inverted" by LU factorization

In [ ]:
# Solve the problem
wh_PC = Function(W)
vpb = LinearVariationalProblem(a, L, wh_PC, bcs=bcs, aP=pc_form)
solver =  LinearVariationalSolver(vpb, nullspace=nsp, solver_parameters=parameters)
t0 = perf_counter()
solver.solve()
print('elapsed time = ', perf_counter() - t0, 's    -    # iter = ', solver.snes.ksp.getIterationNumber())
uh_PC, ph_PC = wh_PC.subfunctions

In [ ]:
fig, ax = plt.subplots()
col = tripcolor(ph_PC, axes=ax)
plt.colorbar(col)
plt.title('pressure')
fig, ax = plt.subplots()
#triplot(mesh, axes=ax)
col = quiver(uh_PC, axes=ax)
plt.colorbar(col)
plt.title('velocity')

### Point 4. Spectra

In [ ]:
# Small problem to investigate eigenvalues
n=5
mesh = UnitSquareMesh(n, n, diagonal='crossed')
V = VectorFunctionSpace(mesh, 'P', 2)
Q = FunctionSpace(mesh, 'P', 1)
W = MixedFunctionSpace([V, Q])
u, p = TrialFunctions(W)
v, q = TestFunctions(W)

bc4 = DirichletBC(W.sub(0), Constant((1., 0.)), 4)
bcRest = DirichletBC(W.sub(0), Constant((0., 0.)), [1,2,3])
bcs = (bc4, bcRest)
# The cavity problem entails a non-trivial nullspace of the matrix, so counting
# the eigenvalues in the clusters is slightly more complex than what theory suggests.
# The clusters expected from the theory can be observed if at least a portion
# of the boundary has Neumann conditions, as in the following Poiseuille problem
# --- uncomment these bcs to change problem ---
# bcs = ( DirichletBC(W.sub(0), Constant((0.,0.)), [3, 4]),
#         DirichletBC(W.sub(0), as_vector((x[1]*(1-x[1]),0.0)), 1)
#        )

# Problem matrix
a = inner(grad(u), grad(v)) * dx - div(v) * p * dx + q * div(u) * dx
L = inner(Constant((0.0,0.0)), v) * dx
Sigma = assemble(a, bcs=bcs).M.values

# Preconditioner
myPCform = inner(grad(u), grad(v)) * dx + p * q * dx
P = assemble(myPCform, bcs=bcs).M.values

ll, vv = np.linalg.eig(Sigma)
true_nullspace = vv[:, abs(ll)<1e-14]

import scipy as sp # numpy
prec_ll, prec_vv = sp.linalg.eig(Sigma, P)
true_nullspace = prec_vv[:, abs(prec_ll)<1e-14]

np.set_printoptions(precision=2)
np.set_printoptions(linewidth=400)
np.set_printoptions(suppress=True)

plt.plot(ll, np.imag(ll), marker='o', linestyle='none', label='eig(Sigma)')
plt.plot(prec_ll, np.imag(prec_ll), marker='+', linestyle='none', label='eig(P^-1 Sigma)')
plt.plot([1, 1], [ min([min(np.imag(ll)), min(np.imag(prec_ll))]), max([max(np.imag(ll)), max(np.imag(prec_ll))]) ], 'k:')
plt.plot([0, 0], [ min([min(np.imag(ll)), min(np.imag(prec_ll))]), max([max(np.imag(ll)), max(np.imag(prec_ll))]) ], 'k:')
plt.plot([ min([min(np.real(ll)), min(np.real(prec_ll))]), max([max(np.real(ll)), max(np.real(prec_ll))]) ], [0, 0], 'k:')
plt.legend()

tol = 1e-10
print('SPECTRUM OF THE ORIGINAL MATRIX')
print(ll.size, 'eigenvalues, of which:')
print('   ', np.sum(abs(ll)<tol), 'equal to 0 (nullspace)')
print('   ', np.sum(abs(ll-1)<tol), 'equal to 1')
print('   ', np.sum(ll < 0), 'negative')
print('   ', np.sum(ll > 0)-np.sum(abs(ll-1)<tol), 'positive != 1')
print('SPECTRUM OF THE PRECONDITIONED MATRIX')
print(prec_ll.size, 'eigenvalues, of which:')
print('   ', np.sum(abs(prec_ll)<tol), 'equal to 0 (dim(nullspace=1))')
print('   ', np.sum(abs(prec_ll-1)<tol), 'equal to 1 (dim(Vh)-dim(Qh) =', V.dim()-Q.dim(),')')
print('   ', np.sum(prec_ll.real < -tol), 'negative (dim(Qh) =', Q.dim(),')')
print('   ', np.sum(prec_ll.real > tol)-np.sum(abs(prec_ll-1)<tol), 'positive != 1 (dim(Qh) =', Q.dim(),')')

### Point 5. Block-triangular preconditioner

Notation for `fieldsplit`

(example with simpler eq. in https://www.firedrakeproject.org/demos/saddle_point_systems.py.html)

$$
\Sigma = \begin{bmatrix} \Sigma_{00} & \Sigma_{01} \\ \Sigma_{10} & \Sigma_{11} \end{bmatrix} = \begin{bmatrix} A & B^T \\ -B & 0 \end{bmatrix}, \qquad P = \begin{bmatrix} P_0 & 0 \\ {\color{red}{P_L}} & P_1 \end{bmatrix} = \begin{bmatrix} A & 0 \\ -B & \frac{1}{\nu}M_p \end{bmatrix}
$$

$$
P\begin{bmatrix} \mathbf{X}_0\\\mathbf{X}_1\end{bmatrix} = \begin{bmatrix} \mathbf{Y}_0\\\mathbf{Y}_1\end{bmatrix}
\quad\rightarrow\quad
\begin{bmatrix} \mathbf{X}_0\\\mathbf{X}_1\end{bmatrix} = \begin{bmatrix} P_0^{-1}\mathbf{Y}_0\\P_1^{-1}(\mathbf{Y}_1-{\color{red}{P_L}}\mathbf{X}_0)\end{bmatrix} = \begin{bmatrix} P_0^{-1}\mathbf{Y}_0\\P_1^{-1}(\mathbf{Y}_1-{\color{red}{P_L}}P_0^{-1}\mathbf{Y}_0)\end{bmatrix}
$$

$$
\Sigma\begin{bmatrix} \mathbf{U} \\\boldsymbol{\Pi} \end{bmatrix}=\begin{bmatrix} \mathbf{F} \\\mathbf{0} \end{bmatrix}
\longrightarrow
P^{-1}\Sigma\begin{bmatrix} \mathbf{U} \\\boldsymbol{\Pi} \end{bmatrix}=P^{-1}\begin{bmatrix} \mathbf{F} \\\mathbf{0} \end{bmatrix}
\leftrightarrow
\begin{bmatrix} P_0^{-1}A & P_0^{-1}B^T \\ P_1^{-1}(-B-{\color{red}{P_L}}P_0^{-1}A) & -P_1^{-1}{\color{red}{P_L}}B^T \end{bmatrix}\begin{bmatrix} \mathbf{U} \\\boldsymbol{\Pi} \end{bmatrix}=\begin{bmatrix} P_0^{-1}\mathbf{F} \\-P_1^{-1}{\color{red}{P_L}}P_0^{-1}\mathbf{F} \end{bmatrix}
$$
so again for $P_0, P_1$ we need to be able to "apply its inverse" (or an approximation of it), whereas for $P_L$ we do not need to invert.

**N.B.** We **never** actually compute the inverse of a matrix: to "apply the inverse" of a matrix means to solve the associated linear system.

In [ ]:
# First, run the cells of points 1-2 defining the problem...

# Create preconditioner form
pc_form = inner(grad(u), grad(v)) * dx + p * q * dx + q*div(u)*dx

parameters = {'ksp_type': 'gmres',
              'ksp_rtol': 1.e-8, 'ksp_max_it': 10000,
              "pc_type": "fieldsplit",                                              # use block structure of monolithic matrix
              "pc_fieldsplit_type": "multiplicative",                               # use a block triangular preconditioner (taken from pc_form)
              "fieldsplit_0_ksp_type": "preonly", "fieldsplit_0_pc_type": "lu",     # block P0 is "inverted" by LU factorization
              "fieldsplit_1_ksp_type": "preonly", "fieldsplit_1_pc_type": "lu"}     # "inverse" of block P1 is approximated by ILU factorization

In [ ]:
# Solve the problem
wh_PC = Function(W)
vpb = LinearVariationalProblem(a, L, wh_PC, bcs=bcs, aP=pc_form)
solver =  LinearVariationalSolver(vpb, nullspace=nsp, solver_parameters=parameters)
t0 = perf_counter()
solver.solve()
print('elapsed time = ', perf_counter() - t0, 's    -    # iter = ', solver.snes.ksp.getIterationNumber())
uh_PC, ph_PC = wh_PC.subfunctions

In [ ]:
fig, ax = plt.subplots()
col = tripcolor(ph_PC, axes=ax)
plt.colorbar(col)
plt.title('pressure')
fig, ax = plt.subplots()
#triplot(mesh, axes=ax)
col = quiver(uh_PC, axes=ax)
plt.colorbar(col)
plt.title('velocity')

---
---
## additional - optimality of the preconditioners

In [ ]:
from math import inf
from time import perf_counter

# Solve cavity problem on (0,1)^2 with fully Dirichlet BC.
# Inputs: n       = mesh subdivisions along each direction
#         params  = solver parameters
def solve_cavity(n, params, create_pc_form=None):
  mesh = UnitSquareMesh(n, n)
  V = VectorFunctionSpace(mesh, 'P', 2)
  Q = FunctionSpace(mesh, 'P', 1)
  W = MixedFunctionSpace([V, Q])
  u, p = TrialFunctions(W)
  v, q = TestFunctions(W)
  bc4 = DirichletBC(W.sub(0), Constant((1., 0.)), 4)
  bcRest = DirichletBC(W.sub(0), Constant((0., 0.)), [1,2,3])
  bcs = (bc4, bcRest)
  a = inner(grad(u), grad(v)) * dx - div(v) * p * dx + q * div(u) * dx
  L = inner(Constant((0.0,0.0)), v) * dx
  nsp = MixedVectorSpaceBasis(
      W, [W.sub(0), VectorSpaceBasis(constant=True,
                                     comm=COMM_WORLD)] # to suppress warnings
  )

  wh = Function(W)
  if (create_pc_form):
    pc_form = create_pc_form(u, p, v, q)
    vpb = LinearVariationalProblem(a, L, wh, bcs, aP=pc_form)
  else:
    vpb = LinearVariationalProblem(a, L, wh, bcs)

  solver =  LinearVariationalSolver(vpb, nullspace=nsp, solver_parameters=params)
  wh.assign(0)
  t0 = perf_counter()
  try:
    solver.solve()
    elapsed = perf_counter() - t0
  except:
    print('NOT ABLE TO SOLVE')
    elapsed = inf

  iter = solver.snes.ksp.getIterationNumber()
  print('||w||=', norm(wh,'H1'), 'elapsed time = ', elapsed, 's    -    # iter = ', iter)

  return elapsed, iter

In [ ]:
# Create preconditioner form
# (we can use the same for additive and multiplicative...)
def create_pc_form(u, p, v, q):
  return inner(grad(u), grad(v)) * dx + p * q * dx + q * div(u) * dx

none_params = {'ksp_type': 'gmres', 'pc_type': 'none',
              'ksp_rtol': 1.e-8, 'ksp_max_it': 10000}

import copy

ilu_params = copy.deepcopy(none_params)
ilu_params.update({'pc_type': 'ilu'})

fs_add_params = copy.deepcopy(none_params)
fs_add_params.update({"pc_type": "fieldsplit",
              "pc_fieldsplit_type": "additive",
              "fieldsplit_0_ksp_type": "preonly", "fieldsplit_0_pc_type": "lu",   # block P0 is "inverted" by LU factorization
              "fieldsplit_1_ksp_type": "preonly", "fieldsplit_1_pc_type": "lu"})  # block P0 is "inverted" by LU factorization

fs_mul_params = copy.deepcopy(fs_add_params)
fs_mul_params.update({"pc_fieldsplit_type": "multiplicative"})

fs_add_ilu0_params = copy.deepcopy(fs_add_params)
fs_add_ilu0_params.update({
    "fieldsplit_0_ksp_type": "preonly", "fieldsplit_0_pc_type": "ilu"})
    # "inverse" of blocks P0 is approximated by its iLU factorization

fs_add_ilu1_params = copy.deepcopy(fs_add_params)
fs_add_ilu1_params.update({
    "fieldsplit_1_ksp_type": "preonly", "fieldsplit_1_pc_type": "ilu"})
    # "inverse" of blocks P1 is approximated by its iLU factorization

prec_strs = ['none', 'ilu',
             'fieldsplit_additive', 'fieldsplit_multiplicative',
             'fieldsplit_additive_ilu0', 'fieldsplit_additive_ilu1']
prec_params = [none_params, ilu_params,
               fs_add_params, fs_mul_params,
               fs_add_ilu0_params, fs_add_ilu1_params]
prec_form_constructors = [None, None,
               create_pc_form, create_pc_form,
               create_pc_form, create_pc_form]

In [ ]:
n_vec = [10, 20, 40, 80]

times = np.zeros((len(prec_strs), len(n_vec)))
iters = np.zeros((len(prec_strs), len(n_vec)))

for ii in range(len(prec_strs)):
  print('\nPRECONDITIONER:', prec_strs[ii])
  print(prec_params[ii])
  for jj in range(len(n_vec)):
    print('\tn =', n_vec[jj])
    times[ii][jj], iters[ii][jj] = solve_cavity(n_vec[jj], prec_params[ii], prec_form_constructors[ii])

print('TIMES', times)
print('ITERS', iters)

In [ ]:
for ii in range(len(prec_strs)):
  print('\nPRECONDITIONER:', prec_strs[ii])
  print('times:', times[ii])
  print('iterations:', iters[ii])
